# VisionPlate ANPR: CRNN-CTC OCR Training Notebook (Kaggle)


This notebook trains an Optical Character Recognition (OCR) model from scratch using a Convolutional Recurrent Neural Network (CRNN) with Connectionist Temporal Classification (CTC) loss.


### Research Paper Reference


The architecture and hyperparameters strictly follow the seminal paper: **"An End-to-End Trainable Neural Network for Image-based Sequence Recognition and Its Application to Scene Text Recognition"** by Baoguang Shi, Xiang Bai, and Cong Yao (IEEE TPAMI, 2015).


**Key Hyperparameters from Paper:**
- Input size: 32x100 grayscale
- CNN: VGG-based feature extractor
- RNN: 2-layer Bidirectional LSTM (Hidden Size: 256)
- Loss: CTC (Connectionist Temporal Classification)
- Optimizer: Adam (Learning Rate: 0.001)
- Batch Size: 64


In [ ]:
!pip install torch torchvision torchaudio opencv-python matplotlib pandas seaborn


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import random
import string

# Ensure reproducibility
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


## 1. Hyperparameters (Based on Shi et al., 2015)


In [ ]:
# --- HYPERPARAMETERS ---
IMG_H = 32
IMG_W = 100
BATCH_SIZE = 64
HIDDEN_SIZE = 256 # BiLSTM hidden size
EPOCHS = 30
LR = 0.001

# Character set (A-Z, 0-9)
ALPHABET = string.ascii_uppercase + string.digits
NUM_CLASSES = len(ALPHABET) + 1 # +1 for CTC Blank token



## 2. Dataset Setup (Dummy Generator for Kaggle)


In a real scenario, you would mount your Kaggle dataset. For this notebook to run out of the box, we use a dynamic synthetic license plate generator.


In [ ]:
class SyntheticPlateDataset(Dataset):
    def __init__(self, num_samples, transform=None):
        self.num_samples = num_samples
        self.transform = transform
        self.alphabet = ALPHABET

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # Generate random Indian plate format like MH12AB1234
        state = ''.join(random.choices(string.ascii_uppercase, k=2))
        dist = ''.join(random.choices(string.digits, k=2))
        letters = ''.join(random.choices(string.ascii_uppercase, k=2))
        nums = ''.join(random.choices(string.digits, k=4))
        text = state + dist + letters + nums
        
        # Create image
        img = np.ones((IMG_H, IMG_W, 3), dtype=np.uint8) * 200 # gray background
        font = cv2.FONT_HERSHEY_SIMPLEX
        cv2.putText(img, text, (5, 22), font, 0.6, (0, 0, 0), 2)
        
        # Add some noise (simulating road conditions)
        noise = np.random.normal(0, 15, img.shape).astype(np.uint8)
        img = cv2.add(img, noise)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        if self.transform:
            img = self.transform(img)
            
        # Encode text to integers for CTC Loss
        encoded_text = [self.alphabet.find(c) + 1 for c in text] # +1 because 0 is blank
        
        return img, torch.tensor(encoded_text, dtype=torch.long), text

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = SyntheticPlateDataset(5000, transform=transform)
val_dataset = SyntheticPlateDataset(1000, transform=transform)

# collate_fn to handle variable length targets
def collate_fn(batch):
    images, targets, texts = zip(*batch)
    images = torch.stack(images, 0)
    target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    targets = torch.cat(targets) # Flatten targets for CTC Loss
    return images, targets, target_lengths, texts

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)



## 3. CRNN Architecture Definition


In [ ]:
class CRNN(nn.Module):
    def __init__(self, img_channels, num_classes, hidden_size):
        super(CRNN, self).__init__()
        
        # CNN (VGG-style based on paper)
        self.cnn = nn.Sequential(
            nn.Conv2d(img_channels, 64, kernel_size=3, padding=1), nn.ReLU(True),
            nn.MaxPool2d(2, 2), # 16x50
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(True),
            nn.MaxPool2d(2, 2), # 8x25
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.ReLU(True),
            nn.MaxPool2d((2, 2), (2, 1), (0, 1)), # 4x26
            nn.Conv2d(256, 512, kernel_size=3, padding=1), nn.BatchNorm2d(512), nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(True),
            nn.MaxPool2d((2, 2), (2, 1), (0, 1)), # 2x27
            nn.Conv2d(512, 512, kernel_size=2, padding=0), nn.BatchNorm2d(512), nn.ReLU(True) # 1x26
        )
        
        # Map CNN output to RNN input
        self.map_to_rnn = nn.Linear(512, hidden_size)
        
        # RNN (2-layer BiLSTM)
        self.rnn1 = nn.LSTM(hidden_size, hidden_size, bidirectional=True, batch_first=True)
        self.rnn2 = nn.LSTM(hidden_size * 2, hidden_size, bidirectional=True, batch_first=True)
        
        # Fully connected to classes
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        # x shape: (batch, 1, 32, 100)
        conv = self.cnn(x) # (batch, 512, 1, W)
        
        # Reshape to (batch, W, 512) for RNN
        b, c, h, w = conv.size()
        conv = conv.squeeze(2)
        conv = conv.permute(0, 2, 1) # (batch, W, 512)
        
        # Sequence modeling
        rnn_in = self.map_to_rnn(conv)
        out, _ = self.rnn1(rnn_in)
        out, _ = self.rnn2(out)
        
        # Predict character probabilities per timestep
        out = self.fc(out) # (batch, W, num_classes)
        
        # PyTorch CTCLoss expects (W, batch, num_classes)
        out = out.permute(1, 0, 2)
        return out

model = CRNN(img_channels=1, num_classes=NUM_CLASSES, hidden_size=HIDDEN_SIZE).to(device)
print(model)


## 4. Loss Function and Optimizer


In [ ]:
criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = optim.Adam(model.parameters(), lr=LR)


## 5. Training Loop


In [ ]:
history = {'loss': [], 'val_loss': []}

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for images, targets, target_lengths, _ in train_loader:
        images = images.to(device)
        targets = targets.to(device)
        
        optimizer.zero_grad()
        preds = model(images) # (W, batch, num_classes)
        
        input_lengths = torch.full(size=(images.size(0),), fill_value=preds.size(0), dtype=torch.long)
        
        loss = criterion(preds.log_softmax(2), targets, input_lengths, target_lengths)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    avg_loss = total_loss / len(train_loader)
    history['loss'].append(avg_loss)
    
    # Validation (Optional)
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for images, targets, target_lengths, _ in val_loader:
            images = images.to(device)
            targets = targets.to(device)
            preds = model(images)
            input_lengths = torch.full(size=(images.size(0),), fill_value=preds.size(0), dtype=torch.long)
            loss = criterion(preds.log_softmax(2), targets, input_lengths, target_lengths)
            val_loss += loss.item()
            
    avg_val_loss = val_loss / len(val_loader)
    history['val_loss'].append(avg_val_loss)
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {avg_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

# Save the trained model
torch.save(model.state_dict(), '/kaggle/working/crnn_plate_reader.pth')
print("Model saved to /kaggle/working/crnn_plate_reader.pth")


## 6. Plot Training Curves


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history['loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.title('CTC Loss over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.savefig('/kaggle/working/ocr_loss_curve.png')
plt.show()


## 7. Decode and Visualize Predictions


In [ ]:
def decode_predictions(preds):
    preds = preds.permute(1, 0, 2) # (batch, W, num_classes)
    _, max_indices = torch.max(preds, 2)
    
    decoded_strings = []
    for seq in max_indices:
        decoded_str = []
        for i in range(len(seq)):
            if seq[i] != 0 and (not (i > 0 and seq[i] == seq[i - 1])):
                decoded_str.append(ALPHABET[seq[i] - 1])
        decoded_strings.append(''.join(decoded_str))
    return decoded_strings

model.eval()
images, _, _, true_texts = next(iter(val_loader))
images = images.to(device)

with torch.no_grad():
    preds = model(images)
    pred_texts = decode_predictions(preds)

# Plot sample predictions
fig, axes = plt.subplots(4, 2, figsize=(12, 10))
axes = axes.flatten()

for i in range(8):
    img = images[i].cpu().squeeze().numpy()
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f"True: {true_texts[i]} | Pred: {pred_texts[i]}", fontsize=10)
    axes[i].axis('off')
    
plt.tight_layout()
plt.savefig('/kaggle/working/ocr_predictions.png')
plt.show()
